# KOA Ontology Bootstrapping

Este notebook lê um arquivo Prolog de um contrato **"ouro"** e gera uma **TBox congelada** para o onboarding do KnowOntoAsk.

Ele gera duas camadas:

1) **TBox estrutural** (predicados/aridades) derivada do Prolog.
2) **Vocabulário congelado**:
   - `allowed_metadata_key/1` (chaves permitidas em `contract_metadata/3`)
   - `allowed_fact_type/1` (tipos permitidos em `contract_clause_fact/5`)

Saída: `KOA_ontology_bootstrap.pl` no diretório configurado.


In [ ]:
# Monta o Google Drive no ambiente Colab para acessar os arquivos de contratos e salvar os .pl gerados
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Importa bibliotecas padrão
import re
from collections import Counter, defaultdict
from pathlib import Path

In [ ]:
# Configura pastas para entradas e saídas
directory = '/content/drive/My Drive/KOA/onboarding/modelos/'
persist_directory = directory

In [ ]:
prolog_input_path = directory + "KOA_195_2022_Brasoftware.pl"  # contrato "ouro" para congelar a TBox
prolog_output_path = persist_directory + "KOA_ontology_bootstrap.pl"

# Base IRI da ontologia (ajustar depois para o IRI oficial do KnowOntoAsk)
base_iri = "http://example.org/koa#"

PROLOG_INPUT_FILE = Path(prolog_input_path)
PROLOG_OUTPUT_FILE = Path(prolog_output_path)

print("Arquivo Prolog de entrada:", PROLOG_INPUT_FILE)
print("Arquivo Prolog de saída:", PROLOG_OUTPUT_FILE)
print("Base IRI:", base_iri)


Arquivo Prolog de entrada: /content/drive/My Drive/KOA/onboarding/modelos/KOA_195_2022_Brasoftware.pl
Arquivo Prolog de saída: /content/drive/My Drive/KOA/onboarding/modelos/KOA_ontology_bootstrap.pl
Base IRI: http://example.org/koa#


In [ ]:
# Funções de apoio

def split_prolog_clauses(text: str):
    """
    Divide o texto Prolog em cláusulas terminadas por '.' fora de strings.
    Tenta respeitar pontos dentro de strings simples ou duplas.
    """
    clauses = []
    buf = []
    in_str = False
    quote = None
    i = 0
    n = len(text)

    while i < n:
        ch = text[i]
        buf.append(ch)

        if in_str:
            if ch == quote:
                in_str = False
            i += 1
            continue

        if ch in ("'", '"'):
            in_str = True
            quote = ch
            i += 1
            continue

        if ch == '.':
            # Fim de cláusula se próximo caractere for espaço, quebra de linha ou fim de arquivo
            j = i + 1
            if j >= n or text[j].isspace():
                clauses.append("".join(buf).strip())
                buf = []
                i += 1
                continue

        i += 1

    rest = "".join(buf).strip()
    if rest:
        clauses.append(rest)

    return clauses

def split_head_args(clause: str):
    """
    Extrai (nome_do_predicado, [args...]) do cabeçalho da cláusula Prolog.
    Ignora comentários e blocos markdown tipo ```prolog.
    """
    clause = clause.strip()

    # Ignora comentários e blocos de markdown
    if clause.startswith("%") or clause.startswith("```"):
        return None

    # Fica só com a parte antes de ':-' (se houver)
    if ':-' in clause:
        head = clause.split(':-', 1)[0].strip()
    else:
        head = clause

    # Remove ponto final, se ainda estiver
    if head.endswith('.'):
        head = head[:-1].strip()

    # Tenta casar padrão predicado(args...)
    m = re.match(r'^([a-z_][A-Za-z0-9_]*)\s*\((.*)\)$', head, flags=re.DOTALL)
    if not m:
        return None

    name = m.group(1)
    args_src = m.group(2).strip()

    def split_args(s: str):
        args = []
        buf = []
        in_str = False
        quote = None
        depth_paren = 0
        depth_brack = 0

        for ch in s:
            if in_str:
                buf.append(ch)
                if ch == quote:
                    in_str = False
                continue

            if ch in ("'", '"'):
                in_str = True
                quote = ch
                buf.append(ch)
                continue

            if ch == '(':
                depth_paren += 1
            elif ch == ')':
                depth_paren -= 1
            elif ch == '[':
                depth_brack += 1
            elif ch == ']':
                depth_brack -= 1

            if ch == ',' and depth_paren == 0 and depth_brack == 0:
                args.append("".join(buf).strip())
                buf = []
            else:
                buf.append(ch)

        rest = "".join(buf).strip()
        if rest:
            args.append(rest)
        return args

    args = split_args(args_src) if args_src else []
    return name, args

def collect_predicates_from_prolog(text: str):
    """
    Percorre cláusulas Prolog e retorna:
      - preds: Counter((nome, aridade) -> ocorrências)
      - examples: alguns exemplos de listas de argumentos (para inspeção)
    """
    clauses = split_prolog_clauses(text)
    preds = Counter()
    examples = defaultdict(list)

    for clause in clauses:
        parsed = split_head_args(clause)
        if not parsed:
            continue
        name, args = parsed
        key = (name, len(args))
        preds[key] += 1
        # guarda alguns exemplos para debug
        if len(examples[key]) < 3:
            examples[key].append(args)

    return preds, examples

In [ ]:
# Teste rápido de leitura (se o arquivo existir)
if PROLOG_INPUT_FILE.exists():
    prolog_text = PROLOG_INPUT_FILE.read_text(encoding="utf-8")
    preds, examples = collect_predicates_from_prolog(prolog_text)
    print(f"Foram encontrados {len(preds)} predicados distintos no modelo Prolog.\n")
    for (name, arity), count in list(preds.items())[:10]:
        print(f"{name}/{arity} - {count} cláusulas. Exemplo de args: {examples[(name, arity)][0]}")
else:
    print("ATENÇÃO: o arquivo Prolog ainda não existe neste ambiente. Faça upload no Colab antes de rodar esta célula.")

Foram encontrados 4 predicados distintos no modelo Prolog.

contract/1 - 1 cláusulas. Exemplo de args: ['contrato_ocs_0195_2022']
contract_metadata/3 - 17 cláusulas. Exemplo de args: ['contrato_ocs_0195_2022', 'numero_ocs', "'0195/2022'"]
contract_clause/4 - 16 cláusulas. Exemplo de args: ['contrato_ocs_0195_2022', 'clausula_primeira_objeto', "'CLÁUSULA PRIMEIRA – OBJETO'", "'O presente Contrato tem por objeto a prestação continuada de serviços de atualização e suporte técnico, na modalidade Software Assurance, do Sistema Gerenciador de Banco de Dados (SGBD) Microsoft SQL Server, conforme especificações constantes do Termo de Referência (Anexo I do Edital do Pregão Eletrônico nº 025/2022 - BNDES) e da proposta apresentada pelo CONTRATADO, respectivamente, Anexos I e II deste Contrato.'"]
contract_clause_fact/5 - 64 cláusulas. Exemplo de args: ['contrato_ocs_0195_2022', 'clausula_primeira_objeto', 'objeto', "'prestação continuada de serviços de atualização e suporte técnico, na modalida

In [ ]:
def generate_prolog_tbox_from_predicates(preds, base_iri: str, module_name: str = "koa_tbox"):
    """
    Gera uma TBox em Prolog (sem dados/indivíduos) contendo:
      - metadados de namespace/base IRI
      - classe contract
      - uma propriedade para cada predicado:
          - aridade 1  -> datatype_property
          - aridade >=2 -> object_property
      Domínio padrão: contract
      Range genérico: xsd_string (unário) ou thing (>=2)
    """

    lines = []
    lines.append("% =====================================================================")
    lines.append(f"% KOA TBox (schema-only) - generated from predicate signatures")
    lines.append("% No individuals (ABox) facts are produced here.")
    lines.append("% =====================================================================")
    lines.append("")

    # Módulo (opcional, mas ajuda a isolar)
    lines.append(f":- module({module_name}, [")
    exports = [
        "base_iri/1",
        "class/1",
        "predicate_signature/2",
        "datatype_property/2",
        "object_property/2",
        "domain/2",
        "range/2",
        "label/2"
    ]
    lines.append("    " + ",\n    ".join(exports))
    lines.append("]).")
    lines.append("")

    # Metadados
    lines.append(f"base_iri('{base_iri}').")
    lines.append("")

    # Classe genérica
    lines.append("% --- Classes ----------------------------------------------------------")
    lines.append("class(contract).")
    lines.append("label(contract, 'Contract').")
    lines.append("")

    # Predicados
    lines.append("% --- Predicates / Properties (schema) --------------------------------")
    lines.append("% Convention:")
    lines.append("%   datatype_property(PredicateName, Arity) for unary predicates")
    lines.append("%   object_property(PredicateName, Arity)   for predicates with arity >= 2")
    lines.append("%   domain(PredicateName/Arity, contract)")
    lines.append("%   range(PredicateName/Arity, xsd_string | thing)")
    lines.append("")

    for (name, arity) in sorted(preds.keys()):
        sig = f"{name}/{arity}"

        # assinatura
        lines.append(f"predicate_signature({name}, {arity}).")
        lines.append(f"label({name}/{arity}, '{sig}').")

        if arity == 1:
            lines.append(f"datatype_property({name}, {arity}).")
            lines.append(f"domain({name}/{arity}, contract).")
            lines.append(f"range({name}/{arity}, xsd_string).")
        else:
            lines.append(f"object_property({name}, {arity}).")
            lines.append(f"domain({name}/{arity}, contract).")
            lines.append(f"range({name}/{arity}, thing).")

        lines.append("")

    return "\n".join(lines)


In [ ]:
# ------------------------------------------------------------------
# Congelamento de vocabulário (TBox "de verdade")
#  - allowed_metadata_key/1
#  - allowed_fact_type/1
# ------------------------------------------------------------------

def sanitize_prolog_text(text: str) -> str:
    """Remove caracteres de controle que podem quebrar o parsing."""
    return re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", text)

def collect_allowed_metadata_keys(text: str):
    # Key é o 2º argumento de contract_metadata(Contract, Key, Value).
    keys = set(re.findall(r"\bcontract_metadata\(\s*[^,]+\s*,\s*([a-zA-Z0-9_]+)\s*,", text))
    return sorted({k.lower() for k in keys})

def collect_allowed_fact_types(text: str):
    # FactType é o 3º argumento de contract_clause_fact(Contract, ClauseId, FactType, Data, Evidence).
    types = set(re.findall(r"\bcontract_clause_fact\(\s*[^,]+\s*,\s*[^,]+\s*,\s*([a-zA-Z0-9_]+)\s*,", text))
    return sorted({t.lower() for t in types})

def generate_frozen_vocab_section(metadata_keys, fact_types) -> str:
    lines = []
    lines.append("% ===============================")
    lines.append("% Frozen vocabulary (TBox)")
    lines.append("% ===============================")
    lines.append("")
    lines.append("% Allowed keys for contract_metadata/3")
    for k in metadata_keys:
        lines.append(f"allowed_metadata_key({k}).")
    lines.append("")
    lines.append("% Allowed fact types for contract_clause_fact/5")
    for t in fact_types:
        lines.append(f"allowed_fact_type({t}).")
    lines.append("")
    lines.append("% Fallback (quando não encaixar no vocabulário congelado):")
    lines.append("%   contract_metadata_raw(ContractId, KeyText, Value, Evidence).")
    lines.append("%   contract_clause_fact_raw(ContractId, ClauseId, FactTypeText, Data, Evidence).")
    lines.append("")
    return "\n".join(lines)

# Executa o bootstrap da ontologia ==> agora congelando também o vocabulário (keys e fact_types)
if not PROLOG_INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Arquivo Prolog não encontrado: {PROLOG_INPUT_FILE}.\n"
        "Faça o upload do arquivo no Colab (ícone de pasta à esquerda) "
        "ou ajuste o caminho em 'prolog_input_path'."
    )

prolog_text = PROLOG_INPUT_FILE.read_text(encoding="utf-8", errors="replace")
prolog_text = sanitize_prolog_text(prolog_text)

preds, examples = collect_predicates_from_prolog(prolog_text)
print(f"Foram encontrados {len(preds)} predicados distintos no modelo Prolog.")

metadata_keys = collect_allowed_metadata_keys(prolog_text)
fact_types = collect_allowed_fact_types(prolog_text)

print(f"Keys de metadata congeladas: {len(metadata_keys)}")
print(f"Fact types congelados: {len(fact_types)}")

tbox_content = generate_prolog_tbox_from_predicates(preds, base_iri=base_iri)
tbox_content += "\n\n" + generate_frozen_vocab_section(metadata_keys, fact_types)

PROLOG_OUTPUT_FILE.write_text(tbox_content, encoding="utf-8")
print(f"TBox congelada gerada em: {PROLOG_OUTPUT_FILE.resolve()}")

Foram encontrados 4 predicados distintos no modelo Prolog.
Keys de metadata congeladas: 17
Fact types congelados: 62
TBox congelada gerada em: /content/drive/My Drive/KOA/onboarding/modelos/KOA_ontology_bootstrap.pl
